# 03. Food Detection Evaluation
This notebook evaluates the performance of the trained YOLO11 food detection model on the unseen validation or test dataset. It outputs standard evaluation metrics (mAP, Precision, Recall, F1-score) and saves them for research documentation.

In [ ]:
import os
import yaml
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

print("Libraries imported.")

## 1. Load Model and Configurations

In [ ]:
# Load app configuration
with open("../configs/app_config.yaml", "r", encoding="utf-8") as f:
    app_config = yaml.safe_load(f)

dataset_dir = Path("..") / app_config["paths"]["dataset_dir"]
data_yaml_path = dataset_dir / "data.yaml"

# Check trained model path
model_path = Path("..") / app_config["paths"]["model_dir"] / "best.pt"

if not model_path.exists():
    print(f"[ERROR] Trained model best.pt not found at: {model_path.resolve()}")
    print("Please train the model or download the best.pt checkpoint first.")
else:
    print(f"Trained model found at: {model_path.resolve()}")
    model = YOLO(str(model_path))
    print("Model loaded successfully.")

## 2. Run Validation on Validation/Test Set

In [ ]:
if model_path.exists() and data_yaml_path.exists():
    print("Evaluating model on validation/test data...")
    # Run evaluation
    metrics = model.val(
        data=str(data_yaml_path),
        split="val", # or "test" if a dedicated test split exists
        imgsz=640,
        device="cpu" # locally we evaluate on CPU
    )
    
    print("\n--- Primary Evaluation Metrics ---")
    print(f"mAP @ 0.5: {metrics.box.map50:.4f}")
    print(f"mAP @ 0.5:0.95: {metrics.box.map:.4f}")
    print(f"Precision (box): {metrics.box.mp:.4f}")
    print(f"Recall (box): {metrics.box.mr:.4f}")
else:
    print("Please ensure model and data.yaml exist before running evaluation.")

## 3. Save Metrics for Research Documentation
We will write the key results to `experiments/food_detection/results.txt` and `metrics.csv` for academic review.

In [ ]:
exp_dir = Path("../experiments/food_detection")
exp_dir.mkdir(parents=True, exist_ok=True)

if 'metrics' in locals():
    # Save CSV metrics
    metrics_data = {
        "precision": [metrics.box.mp],
        "recall": [metrics.box.mr],
        "map50": [metrics.box.map50],
        "map50-95": [metrics.box.map],
        "f1_score": [2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-8)]
    }
    df = pd.DataFrame(metrics_data)
    df.to_csv(exp_dir / "metrics.csv", index=False)
    
    # Save raw results text
    with open(exp_dir / "results.txt", "w", encoding="utf-8") as f:
        f.write("Smart Canteen Food Detection Evaluation Results (Phase 1)\n")
        f.write("="*55 + "\n")
        f.write(f"mAP@0.5:         {metrics.box.map50:.6f}\n")
        f.write(f"mAP@0.5:0.95:    {metrics.box.map:.6f}\n")
        f.write(f"Precision:       {metrics.box.mp:.6f}\n")
        f.write(f"Recall:          {metrics.box.mr:.6f}\n")
        f.write(f"F1 Score:        {metrics_data['f1_score'][0]:.6f}\n")
        
    print(f"Saved metrics in: {exp_dir.resolve()}")
else:
    print("No active metrics found. Skip saving.")